In [ ]:
#!pip install llama-cpp-python

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import gradio as gr
import pandas as pd
import numpy as np
import faiss
import json
import os
import csv
import shutil
from sentence_transformers import SentenceTransformer
from llama_cpp import Llama
from sklearn.metrics.pairwise import cosine_similarity
import re
from datetime import datetime
from paddleocr import PaddleOCR
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from textwrap import wrap
from reportlab.lib.utils import ImageReader

In [3]:
# === File paths ===
base_dir = r"C:\Users\manjo\Downloads\Project"
faiss_path = os.path.join(base_dir, "faiss_index_PaddleOCR.index")
csv_path = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2/chunked_text_PaddleOCR.csv")
text_folder = os.path.join(base_dir, "extracted_text_PaddleOCR2/cleaned_text_PaddleOCR2")
logo_path = os.path.join(base_dir, "Logo.png")
mistral_path = r"C:\Users\manjo\Downloads\mistral-7b-instruct-v0.1.Q4_K_M.gguf"
CACHE_FILE = "cache.json"
FEEDBACK_FILE = "feedback_log.csv"

In [4]:
# === Load components ===
index = faiss.read_index(faiss_path)
df_chunks = pd.read_csv(csv_path)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
llm = Llama(model_path=mistral_path, n_ctx=2048)
ocr_model = PaddleOCR(use_angle_cls=True, lang='en')


llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from C:\Users\manjo\Downloads\mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:  

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Creating model: ('UVDoc', None)
The model(UVDoc) is not supported to run in MKLDNN mode! Using `paddle` instead!
Using official model (UVDoc), the model files will be automatically downloaded and saved in C:\Users\manjo\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

...

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Creating model: ('PP-OCRv5_server_rec', None)
Using official model (PP-OCRv5_server_rec), the model files will be automatically downloaded and saved in C:\Users\manjo\.paddlex\official_models.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

In [5]:
# === Session Logs ===
cache = {}
chat_log = []

# === Cache & feedback helpers ===

def load_cache():
    return json.load(open(CACHE_FILE, "r", encoding="utf-8")) if os.path.exists(CACHE_FILE) else {}

def save_cache(cache):
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=4, ensure_ascii=False)

def save_feedback(query, answer, source, feedback):
    data = {"query": query, "answer": answer, "source": source, "feedback": feedback}
    file_exists = os.path.isfile(FEEDBACK_FILE)
    with open(FEEDBACK_FILE, "a", encoding="utf-8", newline='') as f:
        writer = csv.DictWriter(f, fieldnames=data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(data)

In [6]:
# === Reset everything ===

def reset_all():
    """Clear cache, feedback log, temp docs, and chat state."""
    open(CACHE_FILE, "w").write("{}")
    open(FEEDBACK_FILE, "w").write("query,answer,source,feedback\n")
    shutil.rmtree("temp_docs", ignore_errors=True)
    chat_log.clear()
    # Return placeholders for: chatbot, file_viewer, export_file, export_feedback_file, feedback_table, pdf_file
    return [], None, None, None, None, None

In [7]:
# === OCR extraction ===

def extract_text_from_image(image_path):
    ocr_result = ocr_model.ocr(image_path, cls=True)
    extracted_text = "\n".join([line[1][0] for block in ocr_result for line in block])
    return extracted_text

In [8]:
# === FAISS retrieval (+ optional OCR chunk) ===

def search_top_k(query, k=2, extra_text=None):
    embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(embedding).astype("float32"), k)
    faiss_results = df_chunks.iloc[indices[0]][["chunk_id", "filename", "chunk_text"]]

    if extra_text:
        extra_df = pd.DataFrame([{"chunk_id": "ocr_chunk", "filename": "uploaded_image", "chunk_text": extra_text}])
        faiss_results = pd.concat([faiss_results, extra_df], ignore_index=True)

    return faiss_results

In [9]:
# === Best semantic snippet ===

def find_best_semantic_snippet(chunks_df, question, model, max_length=250):
    question_vec = model.encode([question])[0]
    best_snippet, best_file, best_score = "", "", -1
    for _, row in chunks_df.iterrows():
        sentences = re.split(r'(?<=[.!?]) +', row["chunk_text"])
        for sent in sentences:
            sent = sent.strip()
            if len(sent) < 10:
                continue
            score = cosine_similarity([question_vec], [model.encode([sent])[0]])[0][0]
            if score > best_score:
                best_score = score
                best_snippet = sent
                best_file = row["filename"]
    if not best_snippet:
        return chunks_df.iloc[0]["filename"], chunks_df.iloc[0]["chunk_text"][:max_length].strip() + "..."
    return best_file, best_snippet if len(best_snippet) < max_length else best_snippet[:max_length].strip() + "..."

In [10]:
# === Prompt & answer generation ===

def build_prompt(chunks, question, chat_log=None):
    context = "\n".join(chunks["chunk_text"].tolist())
    history_text = ""
    if chat_log:
        for i, entry in enumerate(chat_log[-2:], start=1):
            history_text += f"Q{i}: {entry['question']}\nA{i}: {entry['answer']}\n"
    prompt = (
        "You are a helpful assistant. Answer the user's question using the provided context.\n\n"
        "### Prior Conversation ###\n" + history_text +
        "### Context from Documents ###\n" + context + "\n\n" +
        "### Question ###\n" + question + "\n\n" +
        "### Answer ###\n"
    )
    return prompt

def generate_answer(chunks, question, chat_log=None):
    response = llm(build_prompt(chunks, question, chat_log), max_tokens=200, stop=["</s>", "###"])
    return response["choices"][0]["text"].strip()


In [11]:
# === Main chat function ===

def chatbot_ui(query, history, image_file):
    cache = load_cache()
    ocr_text = extract_text_from_image(image_file.name) if image_file else None

    if query in cache:
        answer = cache[query]["answer"]
        source = cache[query]["source"]
        snippet = cache[query]["snippet"]
        filepath = os.path.join("temp_docs", source)
    else:
        chunks = search_top_k(query, k=2, extra_text=ocr_text)
        answer = generate_answer(chunks, query, chat_log)
        source, snippet = find_best_semantic_snippet(chunks, query, embedding_model)

        local_file_path = os.path.join(text_folder, source)
        os.makedirs("temp_docs", exist_ok=True)
        filepath = os.path.join("temp_docs", source)
        if os.path.exists(local_file_path):
            shutil.copy(local_file_path, filepath)
        else:
            filepath = None

        cache[query] = {"answer": answer, "source": source, "snippet": snippet}
        save_cache(cache)

    chat_log.append({"question": query, "answer": answer, "source": source, "snippet": snippet})

    display = f"💬 Answer: {answer}\n📄 Source: {source}\n🔍 Snippet: {snippet}"
    history.append((query, display))

    from gradio import update
    file_output = filepath if filepath and os.path.exists(filepath) else update(visible=False)
    return history, query, answer, source, file_output


In [12]:
# === Feedback & CSV export ===

def feedback_fn(query, answer, source, feedback, history):
    save_feedback(query, answer, source, feedback)
    return history + [("✅ Feedback received: " + feedback.upper(), "")]


def export_chat_to_csv():
    from gradio import update
    if not chat_log:
        return update(value=None, visible=False)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"chat_export_{ts}.csv"
    pd.DataFrame(chat_log).to_csv(path, index=False)
    return update(value=path, visible=True)


def export_feedback_to_csv():
    from gradio import update
    if not os.path.exists(FEEDBACK_FILE):
        return update(value=None, visible=False)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    new_path = f"feedback_export_{ts}.csv"
    shutil.copy(FEEDBACK_FILE, new_path)
    return update(value=new_path, visible=True)


In [13]:
# === NEW: Export chat to PDF ===

def export_chat_to_pdf():
    """Generate a nicely formatted PDF of the entire chat_log."""
    from gradio import update
    if not chat_log:
        return update(value=None, visible=False)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_path = f"chat_export_{ts}.pdf"
    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4

    # Header
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, height - 60, "Chatter – Chat Summary Report")
    c.setFont("Helvetica", 10)
    c.drawString(50, height - 75, f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # Logo (optional)
    if os.path.exists(logo_path):
        try:
            c.drawImage(ImageReader(logo_path), width - 100, height - 100, width=40, preserveAspectRatio=True, mask='auto')
        except Exception:
            pass

    # Body
    y = height - 130
    c.setFont("Helvetica", 11)
    for i, entry in enumerate(chat_log, start=1):
        block = [
            f"{i}. Question: {entry['question']}",
            f"   Answer: {entry['answer']}",
            f"   Source: {entry['source']}",
            f"   Snippet: {entry['snippet']}",
        ]
        for line in block:
            for wrap_line in wrap(line, width=100):
                if y < 80:
                    c.showPage()
                    c.setFont("Helvetica", 11)
                    y = height - 80
                c.drawString(50, y, wrap_line)
                y -= 16
            y -= 6
    c.save()
    return update(value=pdf_path, visible=True)


In [14]:
# === Feedback summary loader ===

def load_feedback_summary():
    if not os.path.exists(FEEDBACK_FILE):
        return pd.DataFrame(columns=["query", "answer", "source", "feedback"])
    return pd.read_csv(FEEDBACK_FILE)

# === Gradio UI ===
with gr.Blocks(title="Chatter 1.5 – PDF Enhanced") as demo:
    gr.Markdown("## 🤖 Chatter 1.5 – Complete Chatbot with Feedback & PDF Export")

    chatbot = gr.Chatbot()
    query = gr.Textbox(label="Ask something...")
    image_input = gr.File(label="Upload Image for OCR", file_types=["image"])
    file_viewer = gr.File(label="📄 View Full Document", visible=True)

    # Export / download widgets
    export_csv_btn = gr.Button("💾 Export Chat to CSV")
    export_file = gr.File(label="📥 Download Chat CSV", visible=False)

    export_pdf_btn = gr.Button("🧾 Export Chat to PDF")
    pdf_file = gr.File(label="📄 Download Chat PDF", visible=False)

    export_feedback_btn = gr.Button("📊 Export Feedback Log")
    export_feedback_file = gr.File(label="📥 Download Feedback CSV", visible=False)

    feedback_summary_btn = gr.Button("📈 Show Feedback Summary")
    feedback_table = gr.DataFrame(headers=["query", "answer", "source", "feedback"], interactive=False, visible=False)

    with gr.Row():
        btn_submit = gr.Button("Submit")
        btn_reset = gr.Button("♻️ Reset Chat & Cache")

    with gr.Row():
        btn_good = gr.Button("👍")
        btn_bad = gr.Button("👎")

    # State vars
    state_query = gr.State()
    state_answer = gr.State()
    state_source = gr.State()
    state_file = gr.State()

    # --- Callbacks ---
    btn_submit.click(chatbot_ui, inputs=[query, chatbot, image_input], outputs=[chatbot, state_query, state_answer, state_source, file_viewer])

    btn_reset.click(reset_all, inputs=[], outputs=[chatbot, file_viewer, export_file, export_feedback_file, feedback_table, pdf_file])

    btn_good.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="good"), chatbot], outputs=[chatbot])
    btn_bad.click(feedback_fn, inputs=[state_query, state_answer, state_source, gr.Textbox(value="bad"), chatbot], outputs=[chatbot])

    export_csv_btn.click(export_chat_to_csv, inputs=[], outputs=[export_file])
    export_feedback_btn.click(export_feedback_to_csv, inputs=[], outputs=[export_feedback_file])
    export_pdf_btn.click(export_chat_to_pdf, inputs=[], outputs=[pdf_file])

    feedback_summary_btn.click(load_feedback_summary, inputs=[], outputs=[feedback_table])
    feedback_summary_btn.click(lambda: gr.update(visible=True), None, [feedback_table])

# === Launch ===
if __name__ == "__main__":
    demo.launch()

C:\Users\manjo\AppData\Local\Temp\ipykernel_11004\2067321250.py:12: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\gradio\queueing.py", line 626, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\gradio\route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\gradio\blocks.py", line 2229, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\gradio\blocks.py", line 1740, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\manjo\AppData\Roaming\Python\Python312\site-packages\anyio\to_thr